# 06 - MLflow Model Registry & Monitoring

**Objective:** Extend MLflow from experiment tracking into production — register models in the
Model Registry, manage versions through staging/production transitions, and build a monitoring
pipeline that detects data drift and model degradation using the wine data injection system.

**Concepts Covered:**
- MLflow Tracking Deep-Dive (MlflowClient, programmatic comparison)
- MLflow Model Registry (registration, versioning, stage transitions)
- Population Stability Index (PSI) for prediction drift
- Feature distribution drift (Kolmogorov-Smirnov test)
- Schema drift detection at inference time
- Champion/Challenger deployment pattern
- Automated monitoring loop concept

> **Prerequisite:** This notebook builds on the models trained in `05_model_tuning_validation_comparison.ipynb`.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import datetime
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from scipy.stats import ks_2samp

import mlflow
from mlflow.tracking import MlflowClient
import mlflow.sklearn

import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("All imports successful")


In [ ]:
# TODO: Configure MLflow tracking
#
# 1. Define PROJECT_ROOT = Path("..").resolve()
# 2. Create MLFLOW_DIR = PROJECT_ROOT / "mlflow_artifacts" and mkdir
# 3. Set TRACKING_URI = str(MLFLOW_DIR.absolute())
# 4. mlflow.set_tracking_uri(TRACKING_URI)
# 5. mlflow.set_experiment("wine_06_registry")

PROJECT_ROOT = Path("..").resolve()

# YOUR CODE HERE


In [ ]:
# TODO: Load data, split, scale features
#
# PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
# MODELS_DIR = PROJECT_ROOT / "models"
# df = pd.read_csv(PROCESSED_DIR / "clean_data.csv")
# X = df.drop(columns=["class"])
# y = df["class"]
# FEATURE_NAMES = X.columns.tolist()
#
# X_train, X_test, y_train, y_test = train_test_split(...)
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)
# joblib.dump(scaler, MODELS_DIR / "scaler_monitoring.joblib")

# YOUR CODE HERE


---
## Section 1: Train & Log Models with MLflow

Train the 4 SVC, LogisticRegression, RandomForestClassifier, XGBClassifier and log each
as an MLflow run. Models requiring scaled features use scaled data; tree models use raw.


In [ ]:
# TODO: Train and log baseline models
#
# configs = [
#     ("LogisticRegression", LogisticRegression(max_iter=1000, random_state=42), True),
#     ("RandomForest", RandomForestClassifier(n_estimators=100, random_state=42), False),
#     ("XGBoost", XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss'), False),
#     ("SVM", SVC(probability=True, random_state=42), True),
# ]
# run_ids = {}
#
# mlflow.set_tracking_uri(TRACKING_URI)
# mlflow.set_experiment("wine_06_registry")
#
# for name, model, use_scaled in configs:
#     X_tr = X_train_scaled if use_scaled else X_train
#     X_te = X_test_scaled if use_scaled else X_test
#     model.fit(X_tr, y_train)
#     y_pred = model.predict(X_te)
#     acc = accuracy_score(y_test, y_pred)
#     prec = precision_score(y_test, y_pred, average='macro')
#     rec = recall_score(y_test, y_pred, average='macro')
#     f1 = f1_score(y_test, y_pred, average='macro')
#
#     with mlflow.start_run(run_name=name) as run:
#         mlflow.log_param("model", name)
#         mlflow.log_metric("accuracy", acc)
#         mlflow.log_metric("precision", prec)
#         mlflow.log_metric("recall", rec)
#         mlflow.log_metric("f1", f1)
#         mlflow.sklearn.log_model(model, "model")
#         run_ids[name] = run.info.run_id
#         print(f"{name:20s}  acc={acc:.4f}  f1={f1:.4f}")

# YOUR CODE HERE


---
## Section 2: MLflow Tracking Deep-Dive

Use `MlflowClient` to query experiments programmatically, extract run data into a DataFrame,
and compare models with bar plots.


In [ ]:
# TODO: Query runs with MlflowClient
#
# client = MlflowClient(tracking_uri=TRACKING_URI)
# experiment = client.get_experiment_by_name("wine_06_registry")
# runs = client.search_runs(experiment_ids=[experiment.experiment_id], order_by=["metrics.f1_macro DESC"])
#
# for run in runs:
#     name = run.data.tags.get("mlflow.runName", "unnamed")
#     print metrics

# YOUR CODE HERE


In [ ]:
# TODO: Extract runs → DataFrame → bar plot
#
# Build rows list from runs, convert to DataFrame, display.
# Plot bar chart comparing metrics by model.

# YOUR CODE HERE


---
## Section 3: MLflow Model Registry

The **Model Registry** groups models under a registered name, assigns versions,
and manages stages: `None → Staging → Production → Archived`.

> Note: We load by version number (`models:/name/1`), not stage alias (`/Production`),
> because file-based MLflow tracking doesn't support stage-based loading.


In [ ]:
# TODO: Register models in the Model Registry
#
# REGISTERED_MODEL_NAME = "wine_classifier"
# for name, run_id in run_ids.items():
#     result = mlflow.register_model(f"runs:/{run_id}/model", REGISTERED_MODEL_NAME)
#     print(f"Registered {name:20s} → version {result.version}")

# YOUR CODE HERE


In [ ]:
# TODO: Set version stages with MlflowClient
#
# versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
# run_to_version = {v.run_id: v for v in versions}
#
# # Best f1_macro → Production
# best_version = run_to_version[runs[0].info.run_id]
# client.transition_model_version_stage(...)
#
# # Second best → Staging
# second_version = run_to_version[runs[1].info.run_id]
# client.transition_model_version_stage(...)
#
# # Re-fetch, archive the rest
# versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
# for v in versions:
#     if v.current_stage not in ("Production", "Staging"):
#         client.transition_model_version_stage(...)

# YOUR CODE HERE


In [ ]:
# TODO: Load Production model by version number
#
# PRODUCTION_VERSION = best_version.version
# prod_model = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL_NAME}/{PRODUCTION_VERSION}")
# ref_accuracy = accuracy_score(y_test, prod_model.predict(X_test_scaled))
# ref_proba = prod_model.predict_proba(X_test_scaled)[:, 1]
# print(f"Production model accuracy: {ref_accuracy:.4f}")

# YOUR CODE HERE


---
## Section 4: Simulated Monitoring — Data Drift Detection

Evaluate the Production model against 7 pre-generated corrupted variants
(in `data/processed/corrupted/`). Track accuracy drop, PSI, and feature KS-statistics.


In [ ]:
# TODO: Load corrupted datasets
#
# CORRUPTED_DIR = PROCESSED_DIR / "corrupted"
# preset_names = ["missing_light", "missing_heavy", "noise_low", "noise_high",
#                 "outliers", "bias", "schema_drift"]
# corrupted_data = {}
# for preset in preset_names:
#     corrupted_data[preset] = pd.read_csv(CORRUPTED_DIR / f"corrupted_{preset}.csv")
#     print(f"{preset:20s}: {corrupted_data[preset].shape}")

# YOUR CODE HERE


In [ ]:
# TODO: Implement PSI and feature drift functions
#
# def compute_psi(expected, actual, bins=10):
#     expected = np.clip(expected, 0.001, 0.999)
#     actual = np.clip(actual, 0.001, 0.999)
#     breaks = np.linspace(0, 1, bins + 1)
#     ep = np.histogram(expected, breaks)[0] / len(expected)
#     ap = np.histogram(actual, breaks)[0] / len(actual)
#     ep = np.where(ep == 0, 0.001, ep)
#     ap = np.where(ap == 0, 0.001, ap)
#     return np.sum((ap - ep) * np.log(ap / ep))
#
# def compute_feature_drift(ref_df, corr_df, max_cols=5):
#     columns = ref_df.select_dtypes(include=[np.number]).columns[:max_cols]
#     drifts = {}
#     for col in columns:
#         if col in corr_df.columns:
#             stat, pval = ks_2samp(ref_df[col].dropna(), corr_df[col].dropna())
#             drifts[col] = {"ks_stat": stat, "p_value": pval}
#     return drifts

# YOUR CODE HERE


In [ ]:
# TODO: Run drift evaluation for each corrupted preset
#
# mlflow.set_experiment("wine_06_monitoring")
# monitoring_results = []
#
# For each preset:
#   1. Detect target column
#   2. Align feature columns (add missing with 0, select only FEATURE_NAMES)
#   3. Drop NaN target rows
#   4. Scale features
#   5. Predict with prod_model
#   6. Compute metrics, accuracy_drop, psi, avg_ks_stat
#   7. Start an MLflow run, log all metrics
#
# Use try/except to catch schema drift and other errors gracefully

# YOUR CODE HERE


In [ ]:
# TODO: Visualize monitoring results
#
# Convert monitoring_results to DataFrame, display it.
# Create multi-panel figure with accuracy_drop, psi, avg_ks.

# YOUR CODE HERE


In [ ]:
# TODO: Handle schema drift explicitly
#
# Load corrupted_schema_drift.csv. Check if "class" column exists.
# If not, find the actual target column and print a structured alert.

# YOUR CODE HERE


---
## Section 5: Champion/Challenger Pattern

Train an improved challenger model, register it, stage it as Staging,
and compare its robustness against the Production champion on corrupted data.


In [ ]:
# TODO: Train a challenger, register, and stage
#
# challenger = XGBClassifier(n_estimators=200, max_depth=7, learning_rate=0.15,
#     random_state=42, eval_metric='mlogloss')
# challenger.fit(X_train, y_train)
#
# with mlflow.start_run(run_name="Challenger_XGBoost") as run:
#     mlflow.sklearn.log_model(challenger, "model")
#     challenger_run_id = run.info.run_id
#
# result = mlflow.register_model(f"runs:/{challenger_run_id}/model", REGISTERED_MODEL_NAME)
# CHALLENGER_VERSION = result.version
# client.transition_model_version_stage(name=REGISTERED_MODEL_NAME,
#     version=CHALLENGER_VERSION, stage="Staging")

# YOUR CODE HERE


In [ ]:
# TODO: Compare champion vs challenger on all corrupted presets
#
# champion_model = prod_model
# challenger_model = mlflow.sklearn.load_model(...)
#
# For each preset: preprocess, predict with both, record metrics, determine winner
# Plot grouped bar chart

# YOUR CODE HERE


In [ ]:
# TODO: Decision gate — promote if challenger wins >50%
#
# wins = comp_df["challenger_wins"].sum()
# total = len(comp_df)
# print(f"Challenger wins on {wins}/{total} scenarios")
#
# if wins > total / 2:
#     client.transition_model_version_stage(...)
#     print("Challenger promoted!")
# else:
#     print("Champion retains Production.")

# YOUR CODE HERE


---
## Section 6: Automated Monitoring Loop (Conceptual)

Simulate a 5-day monitoring timeline. Each day brings a different corruption type.
Track metrics and PSI over time, and trigger alerts when performance degrades significantly.


In [ ]:
# TODO: Simulate daily monitoring over 5 days
#
# daily_presets = ["missing_light", "noise_low", "noise_high", "outliers", "bias"]
# timeline = []
#
# for day, preset in enumerate(daily_presets, start=1):
#     Load, preprocess, predict, compute metrics
#     alert based on threshold
#     mlflow.log_metric(...)
#
# Dual-axis plot: metric (left) + PSI (right) over days

# YOUR CODE HERE


---
## Summary

1. **MlflowClient API**: Query experiments, extract runs, compare programmatically
2. **Model Registry**: Register models, assign versions, transition through stages
3. **Drift Monitoring**: PSI, KS-test, accuracy drop — and how to interpret thresholds
4. **Schema Drift**: Detecting column mismatches at inference time
5. **Champion/Challenger**: Systematic model comparison and promotion decisions
6. **Automated Monitoring**: Scheduled evaluation loops with alerting


---
## Exercises

1. **Alerting System**: Define `check_alerts(monitoring_df, ...)` that returns triggered alerts.
2. **Multi-Model Monitoring**: Register each model as separate registered names. Which is most robust?
3. **Feature Importance Drift**: For champion and challenger (tree-based), extract feature importances.
4. **Custom Corruption**: Add a new preset, regenerate corrupted data, re-run monitoring.
5. **Auto-Retraining**: Build a self-healing pipeline: monitor → detect drift → retrain → compare → promote.
